# Coastal flood step 01b: buildings preclean intersections comparison

Purpose:
- keep original network/buildings files unchanged
- run an alternative intersections pass for buildings only using a copied precleaned buildings layer
- compare baseline vs precleaned intersections outputs


In [ ]:
import subprocess
from pathlib import Path

import geopandas as gpd
import pandas as pd

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
data_root = base_path / "dphil_common_cross_cutting/common_incoming_data"

# Baseline (existing) intersections outputs
baseline_output_path = base_path / "dphil_paper_3/results/coastal_flood_network_intersections"
baseline_network_csv = baseline_output_path / "network_layers_for_intersections.csv"
baseline_hazard_csv = baseline_output_path / "coastal_flood_rasters_for_intersections.csv"

# New comparison workspace
compare_root = base_path / "dphil_paper_3/results/coastal_flood_network_intersections_buildings_preclean_compare"
preclean_input_path = compare_root / "input_variants"
preclean_output_path = compare_root / "preclean_intersections"
preclean_input_path.mkdir(parents=True, exist_ok=True)
preclean_output_path.mkdir(parents=True, exist_ok=True)

original_buildings_gpkg = data_root / "networks/networks/buildings/buildings_assigned_economic_activity.gpkg"

if not baseline_hazard_csv.exists():
    raise FileNotFoundError(f"Missing baseline hazard csv: {baseline_hazard_csv}")
if not original_buildings_gpkg.exists():
    raise FileNotFoundError(f"Missing original buildings gpkg: {original_buildings_gpkg}")

print("Baseline intersections:", baseline_output_path)
print("Comparison workspace:", compare_root)


In [ ]:
# Create a copied precleaned buildings file (original file is untouched)
buildings = gpd.read_file(original_buildings_gpkg, layer="areas")
orig_count = len(buildings)
orig_area_m2 = float(buildings.to_crs("EPSG:3448").geometry.area.sum())

buildings["geometry"] = buildings.geometry.make_valid()
buildings = buildings[buildings.geometry.notna() & (~buildings.geometry.is_empty)].copy()

clean_count = len(buildings)
clean_area_m2 = float(buildings.to_crs("EPSG:3448").geometry.area.sum())

preclean_buildings_gpkg = preclean_input_path / "buildings_assigned_economic_activity.gpkg"
if preclean_buildings_gpkg.exists():
    preclean_buildings_gpkg.unlink()

buildings.to_file(preclean_buildings_gpkg, layer="areas", driver="GPKG")

# Build a buildings-only network csv for intersections
if baseline_network_csv.exists():
    network_layers_table = pd.read_csv(baseline_network_csv)
else:
    network_layers_input_file = data_root / "networks/network_layers_hazard_intersections_details.csv"
    network_layers_table = pd.read_csv(network_layers_input_file)[["path"]].drop_duplicates().reset_index(drop=True)
    network_layers_table["path"] = network_layers_table["path"].str.replace(r"^networks/", "networks/networks/", regex=True)

buildings_mask = network_layers_table["path"].str.contains("buildings/buildings_assigned_economic_activity.gpkg", regex=False)
buildings_only_network_csv = network_layers_table.loc[buildings_mask, ["path"]].copy()
if buildings_only_network_csv.empty:
    raise RuntimeError("Could not find buildings_assigned_economic_activity.gpkg in network layers table")

# Use absolute path to precleaned copy so original network files are untouched
buildings_only_network_csv.loc[:, "path"] = str(preclean_buildings_gpkg)
preclean_network_csv = preclean_input_path / "network_layers_for_intersections_buildings_only_preclean.csv"
buildings_only_network_csv.to_csv(preclean_network_csv, index=False)

# Copy hazard csv locally so this variant run does not touch baseline transform files
preclean_hazard_csv = preclean_input_path / "coastal_flood_rasters_for_intersections.csv"
pd.read_csv(baseline_hazard_csv).to_csv(preclean_hazard_csv, index=False)

print(f"Original features: {orig_count:,}")
print(f"Precleaned features: {clean_count:,}")
print(f"Original total area (m2): {orig_area_m2:,.2f}")
print(f"Precleaned total area (m2): {clean_area_m2:,.2f}")
print("Precleaned buildings file:", preclean_buildings_gpkg)
print("Preclean network csv:", preclean_network_csv)
print("Preclean hazard csv:", preclean_hazard_csv)


In [ ]:
# Run alternative intersections for buildings only
run_preclean_intersections = True

if run_preclean_intersections:
    args = [
        "python",
        str(base_path / "robyns_libraries/vector_raster_intersections.py"),
        str(preclean_network_csv),
        str(preclean_hazard_csv),
        str(preclean_output_path),
    ]
    print("Running:", args)
    run_result = subprocess.run(args, capture_output=True, text=True)
    print("Return code:", run_result.returncode)
    print("SKIPPING GEOMETRY count:", run_result.stdout.count("SKIPPING GEOMETRY with error"))
    print("Done marker found:", "Done." in run_result.stdout)
    if run_result.returncode != 0:
        print("STDOUT tail:\n" + "\n".join(run_result.stdout.splitlines()[-60:]))
        print("STDERR tail:\n" + "\n".join(run_result.stderr.splitlines()[-60:]))
        raise RuntimeError("Preclean intersections run failed")
else:
    print("Skipped run. Set run_preclean_intersections = True to run.")


In [ ]:
# Compare baseline vs precleaned buildings intersections outputs
hazard_slug = preclean_hazard_csv.stem
baseline_buildings_intersections = baseline_output_path / f"buildings_assigned_economic_activity_splits__{hazard_slug}__areas.geoparquet"
preclean_buildings_intersections = preclean_output_path / f"buildings_assigned_economic_activity_splits__{hazard_slug}__areas.geoparquet"

if not baseline_buildings_intersections.exists():
    raise FileNotFoundError(f"Missing baseline buildings intersections file: {baseline_buildings_intersections}")
if not preclean_buildings_intersections.exists():
    raise FileNotFoundError(f"Missing preclean buildings intersections file: {preclean_buildings_intersections}")

baseline_gdf = gpd.read_parquet(baseline_buildings_intersections)
preclean_gdf = gpd.read_parquet(preclean_buildings_intersections)

hazard_cols = sorted([c for c in baseline_gdf.columns if c.startswith("coastal_")])

def summarize(gdf, label):
    s = {
        "variant": label,
        "rows": int(len(gdf)),
        "area_m2": float(gdf.to_crs("EPSG:3448").geometry.area.sum()),
    }
    if "osm_id" in gdf.columns:
        s["unique_osm_id"] = int(gdf["osm_id"].nunique())
    if hazard_cols:
        for c in hazard_cols:
            s[f"sum::{c}"] = float(gdf[c].fillna(0).sum())
    return s

summary_df = pd.DataFrame([
    summarize(baseline_gdf, "baseline"),
    summarize(preclean_gdf, "preclean"),
]).set_index("variant")

delta_series = (summary_df.loc["preclean"] - summary_df.loc["baseline"]).rename("delta_preclean_minus_baseline")
display(summary_df)
display(delta_series.to_frame())

print("Baseline file:", baseline_buildings_intersections)
print("Preclean file:", preclean_buildings_intersections)
